# 08 - Neighbourhood processing: smoothing and sharpening

**Syllabus:** neighbourhood processing · image smoothing and sharpening · lightweight
smoothing/sharpening filters for resource-constrained devices

Assignments 1-4 were **point** operations: output pixel from input pixel. This notebook is the
step up to **neighbourhood** operations, where the output depends on a window around the pixel.
Everything downstream — edges, corners, SIFT, and every convolutional layer of a CNN — is this
one operation with different numbers in the kernel.

In [ ]:
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
# the graders are the test harness -- everything else is written out in full
from cvlab import check_convolve, check_gaussian_kernel, check_median, check_unsharp

g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)

## Correlation vs convolution

Slide a `k x k` kernel over the image. At each position, multiply overlapping values and sum.
That is **correlation**. **Convolution** is the same thing with the kernel rotated 180° first:

$$(f * w)(x,y) = \sum_{s}\sum_{t} w(s,t)\, f(x-s, y-t)
\qquad\text{vs}\qquad
(f \otimes w)(x,y) = \sum_{s}\sum_{t} w(s,t)\, f(x+s, y+t)$$

Note the **minus** signs. For a symmetric kernel (box, Gaussian, Laplacian) they give identical
results, which is why the distinction seems academic — until you use an asymmetric kernel like
a gradient, and your edges come out with the sign flipped.

Convolution is the one with the good algebra: commutative, associative, and it becomes plain
multiplication in the Fourier domain (notebook 10). `cv2.filter2D` actually computes
**correlation**, despite the name. So does every "convolutional" neural network.

### Task 1 - convolution from scratch

Zero padding, `same` output size, **true convolution** (flip the kernel). Return **float**, not
`uint8` — a gradient kernel produces negative values and the caller decides what to do with them.
Clipping inside the function would destroy exactly the information the next stage needs.

In [ ]:
def my_filter2D(gray, kernel):
    """True 2-D convolution with zero padding and same-size output.

    Parameters
    ----------
    gray   : (H, W) uint8 array
    kernel : (kh, kw) float array, odd side lengths

    Returns
    -------
    (H, W) float64 array. NOT clipped and NOT cast to uint8 -- negative values must survive.
    A kernel that is 1 at the centre and 0 elsewhere returns the input unchanged.
    """
    # TODO: pad, flip the kernel, then accumulate one shifted-and-scaled copy per kernel tap
    raise NotImplementedError

In [ ]:
check_convolve(my_filter2D)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.filter2D(src, ddepth, kernel, anchor=, delta=, borderType=)
#   ddepth=-1 keeps the input depth (and CLIPS!). Use cv2.CV_64F to keep negatives.
#   IMPORTANT: filter2D computes CORRELATION, not convolution. Flip the kernel yourself.
import cv2, numpy as np
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
asym = np.array([[0,0,0],[1,0,0],[0,0,0]], float)
corr = cv2.filter2D(g.astype(np.float64), -1, asym, borderType=cv2.BORDER_CONSTANT)
conv = cv2.filter2D(g.astype(np.float64), -1, np.flip(asym), borderType=cv2.BORDER_CONSTANT)
print("your convolve vs cv2 correlation      : max diff",
      np.abs(my_filter2D(g, asym) - corr).max(), " <- should be LARGE")
print("your convolve vs cv2 with flipped kern: max diff",
      np.abs(my_filter2D(g, asym) - conv).max(), " <- should be ~0")
print("\nRelated one-liners you should not hand-roll in real code:")
print("  cv2.blur(src, ksize)                 box blur")
print("  cv2.GaussianBlur(src, ksize, sigmaX) ksize=(0,0) derives size from sigma")
print("  cv2.sepFilter2D(src, ddepth, kx, ky) separable, much faster")
print("  cv2.Sobel / cv2.Laplacian / cv2.Scharr  named derivative filters")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

The slow way is two nested loops over *pixels* with an inner sum — correct but painfully
slow in Python. The fast way loops over the **kernel taps** instead (only k² iterations)
and does whole-array arithmetic each time:

```
ph, pw = kh // 2, kw // 2
src = np.pad(gray.astype(np.float64), ((ph, ph), (pw, pw)))   # zero padding
kf  = np.flip(kernel)                                          # <- makes it convolution
out = np.zeros(gray.shape)
for i in range(kh):
    for j in range(kw):
        out += kf[i, j] * src[i:i+H, j:j+W]
```

`np.flip` with no axis argument reverses **both** axes, which is the 180° rotation you want.
Write the loop version first if the sliding-window view is not obvious yet.

</details>

In [ ]:
# The kernel zoo. Same operation, different numbers.
kernels = {
    "box 5x5":      np.ones((5,5))/25,
    "gaussian 5x5": cv2.getGaussianKernel(5,1.0) @ cv2.getGaussianKernel(5,1.0).T,
    "sharpen":      np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], float),
    "laplacian":    np.array([[0,1,0],[1,-4,1],[0,1,0]], float),
    "sobel x":      np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float),
    "emboss":       np.array([[-2,-1,0],[-1,1,1],[0,1,2]], float),
}
outs, names = [], []
for n, k in kernels.items():
    o = my_filter2D(g, k)
    # signed results are shown offset by 128 so negatives are visible
    outs.append(np.clip(o + (128 if o.min() < -1 else 0), 0, 255).astype(np.uint8)); names.append(n)
_imgs, _titles = [g] + list(outs), ["input"] + names
_cols = 7; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

### Task 2 - the Gaussian kernel

Smoothing kernels are not arbitrary. The Gaussian is the one you want almost always: it is the
only kernel that is **separable**, **circularly symmetric**, and introduces **no new extrema**
(no ringing). Its two knobs are the window size `k` and the width `sigma`.

$$w(s,t) = \frac{1}{Z}\exp\!\left(-\frac{s^2+t^2}{2\sigma^2}\right)$$

where `Z` normalises the weights to sum to 1 — otherwise the image gets brighter or darker.
Coordinates `s, t` are measured from the **centre** tap.

In [ ]:
def my_getGaussianKernel(k, sigma):
    """A normalised 2-D Gaussian kernel.

    Parameters
    ----------
    k     : odd int, side length
    sigma : float > 0, standard deviation in pixels

    Returns
    -------
    (k, k) float array summing to exactly 1, symmetric about the centre, all weights positive.
    Must equal the outer product of cv2.getGaussianKernel(k, sigma) with itself.
    """
    # TODO: 1-D coordinates centred on zero, 1-D Gaussian, outer product, normalise
    raise NotImplementedError

In [ ]:
check_gaussian_kernel(my_getGaussianKernel)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.getGaussianKernel(ksize, sigma) -> (ksize, 1) COLUMN vector, 1-D only.
# The 2-D kernel is the outer product -- which is the definition of separable.
import cv2, numpy as np
k1 = cv2.getGaussianKernel(5, 1.0)
print("cv2.getGaussianKernel(5, 1.0).T =", k1.ravel().round(4))
print("shape:", k1.shape, "(a column, not a row -- transpose it to print)")
assert np.allclose(k1 @ k1.T, my_getGaussianKernel(5, 1.0), atol=1e-9)
print("\nouter product matches your 2-D kernel.")
print("If sigma <= 0, cv2 derives it from ksize as 0.3*((ksize-1)*0.5 - 1) + 0.8")
print("cv2.GaussianBlur(src, (0,0), sigma) goes the other way: size derived from sigma.")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Build it 1-D and take an outer product — that is what separability *means*:

`a = np.arange(k) - k // 2` gives centred coordinates like [-2,-1,0,1,2].
`gv = np.exp(-(a**2) / (2*sigma**2))`, then `K = np.outer(gv, gv)`, then `K /= K.sum()`.

Normalising **after** the outer product is what makes the sum exactly 1. You do not need
the 1/(2*pi*sigma^2) constant at all — the normalisation absorbs it.

</details>

In [ ]:
# Separability is a real speedup: 2*k multiplies per pixel instead of k*k.
K = my_getGaussianKernel(15, 3.0)
gv = cv2.getGaussianKernel(15, 3.0)
big = cv2.resize(g, (1024, 1024))
t = lambda f, n=5: (lambda s: (f(), (time.perf_counter()-s)/1*1000)[1])(time.perf_counter())
print("rank of the 15x15 Gaussian:", np.linalg.matrix_rank(K, tol=1e-10), "(1 = separable)")
for label, fn in [("full 2-D filter2D  ", lambda: cv2.filter2D(big, -1, K)),
                  ("separable sepFilter2D", lambda: cv2.sepFilter2D(big, -1, gv, gv)),
                  ("GaussianBlur         ", lambda: cv2.GaussianBlur(big, (15,15), 3.0)),
                  ("3x box blur (approx) ", lambda: cv2.blur(cv2.blur(cv2.blur(big,(9,9)),(9,9)),(9,9)))]:
    for _ in range(3): fn()
    s = time.perf_counter()
    for _ in range(10): fn()
    print(f"{label} {(time.perf_counter()-s)/10*1000:6.2f} ms")

That last row is the **resource-constrained** trick your syllabus asks about: three box blurs
approximate a Gaussian (central limit theorem), and a box blur can be computed in **O(1) per
pixel regardless of radius** using a running sum or an integral image. No multiplies, no
kernel table. That is what runs on a microcontroller.

In [ ]:
approx = cv2.blur(cv2.blur(cv2.blur(g,(9,9)),(9,9)),(9,9))
exact  = cv2.GaussianBlur(g, (0,0), 4.4)
_imgs, _titles = [g, exact, approx, np.abs(exact.astype(int)-approx.astype(int)).astype(np.uint8)*8], ["input","true Gaussian sigma=4.4","3x box 9x9","|difference| x8"]
fig, axes = plt.subplots(1, 4, figsize=(3.4 * 4, 3.4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
print("mean absolute difference:", np.abs(exact.astype(int)-approx.astype(int)).mean().round(2), "levels")

### Task 3 - the median filter

Not a convolution at all — there is no kernel and no weighted sum. It is an **order-statistic**
filter: sort the window, take the middle. That non-linearity is why it removes salt-and-pepper
noise that any linear filter merely smears around, and why it preserves edges that a Gaussian
rounds off.

Use **replicate** padding (repeat the edge pixel) so your result matches `cv2.medianBlur` exactly.
`k=1` returns the input.

In [ ]:
def my_medianBlur(gray, k):
    """k x k median filter with replicate (edge-repeat) padding.

    Parameters
    ----------
    gray : (H, W) uint8 array
    k    : odd int >= 1

    Returns
    -------
    (H, W) uint8 array. Must match cv2.medianBlur(gray, k) exactly.
    """
    # TODO: pad with mode="edge", then take the median of each k x k window
    raise NotImplementedError

In [ ]:
check_median(my_medianBlur)

In [ ]:
# ---- The real OpenCV call ----------------------------------------------------
# cv2.medianBlur(src, ksize) -- ksize must be ODD. For ksize > 5 the input must be uint8.
import cv2, numpy as np
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)[:90, :110]
assert np.array_equal(cv2.medianBlur(g, 5), my_medianBlur(g, 5)), "disagrees with cv2"
print("match (and cv2 is ~1000x faster -- it uses a running histogram).")
print("\nThe rest of the smoothing family:")
print("  cv2.blur / cv2.boxFilter      box average")
print("  cv2.GaussianBlur              weighted average")
print("  cv2.medianBlur                order statistic, kills salt-and-pepper")
print("  cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace)   edge-preserving")

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

`src = np.pad(gray, k//2, mode="edge")` handles the border, and then two loops over
output pixels with `np.median(src[i:i+k, j:j+k])` is correct and readable. It is slow —
that is expected here, and the grader only uses small crops.

Handle `k == 1` up front by returning a copy, or the padding of zero width will bite you.

</details>

In [ ]:
# Salt-and-pepper is the median's home turf.
rng = np.random.default_rng(0)
crop = g[:150, :150]
noisy = crop.copy()
sp = rng.random(crop.shape)
noisy[sp < 0.06] = 0; noisy[sp > 0.94] = 255
_imgs, _titles = [crop, noisy, my_medianBlur(noisy, 3), cv2.GaussianBlur(noisy, (3,3), 0), cv2.blur(noisy,(3,3))], ["clean","6% salt & pepper","median 3x3","gaussian 3x3","box 3x3"]
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()
err = lambda x: np.abs(x.astype(int)-crop.astype(int)).mean()
print(f"mean abs error   median {err(my_medianBlur(noisy,3)):5.2f}   "
      f"gaussian {err(cv2.GaussianBlur(noisy,(3,3),0)):5.2f}   box {err(cv2.blur(noisy,(3,3))):5.2f}")

### Task 4 - unsharp masking

Sharpening is subtracting a blurred version — you cannot *add* detail that is not there, only
amplify what already is. Precisely:

$$g_{\text{sharp}} = f + \alpha\,(f - \text{blur}(f))$$

The bracket is the **mask**: high frequencies only. `α = 1` is unsharp masking; `α > 1` is
high-boost filtering. Use `cv2.GaussianBlur(gray, (0, 0), sigma)` for the blur (the `(0,0)` lets
OpenCV pick the window from sigma), clip to `[0, 255]`, and round.

In [ ]:
def my_unsharp_mask(gray, sigma, amount):
    """Unsharp masking / high-boost filtering.

    out = clip(gray + amount * (gray - GaussianBlur(gray, (0,0), sigma)), 0, 255)

    Parameters
    ----------
    gray   : (H, W) uint8 array
    sigma  : float > 0, blur radius of the mask
    amount : float >= 0. 0 returns the input; 1 is classic unsharp; >1 is high-boost.

    Returns
    -------
    (H, W) uint8 array, rounded and clipped. A flat region is returned unchanged.
    """
    # TODO: float maths, then clip, round, cast
    raise NotImplementedError

In [ ]:
check_unsharp(my_unsharp_mask)

In [ ]:
# ---- The real OpenCV idiom ---------------------------------------------------
# There is no cv2.unsharpMask. The idiom is cv2.addWeighted:
#   cv2.addWeighted(src1, alpha, src2, beta, gamma) = alpha*src1 + beta*src2 + gamma
# Unsharp masking is exactly alpha = 1+amount, beta = -amount:
import cv2, numpy as np
g = cv2.imread("images/lena.jpg", cv2.IMREAD_GRAYSCALE)
sigma, amount = 1.5, 1.0
blur = cv2.GaussianBlur(g, (0, 0), sigma)
cv_out = cv2.addWeighted(g, 1 + amount, blur, -amount, 0)
mine = my_unsharp_mask(g, sigma, amount)
print("max difference vs yours:", int(np.abs(cv_out.astype(int) - mine.astype(int)).max()),
      "(addWeighted saturates and rounds slightly differently)")
print("\nWhy addWeighted and not plain arithmetic? It SATURATES instead of wrapping,")
print("so you cannot get the uint8 overflow bug at all. Same for cv2.add / cv2.subtract.")
print("np.uint8(250) + np.uint8(10) =", np.uint8(250) + np.uint8(10),
      "  vs  cv2.add:", cv2.add(np.uint8([[250]]), np.uint8([[10]]))[0,0])

<details>
<summary><b>Nudge (open only after you have tried)</b></summary>

Cast to float **before** the subtraction or the `uint8` will wrap where the blur exceeds
the original — that is the trap from notebook 01 in a new costume.

`np.clip(f + amount*(f - blur), 0, 255).round().astype(np.uint8)`.

</details>

In [ ]:
soft = cv2.GaussianBlur(g, (0,0), 1.6)          # pretend this is a slightly out-of-focus shot
_imgs, _titles = [soft, my_unsharp_mask(soft,1.0,0.5), my_unsharp_mask(soft,1.0,1.5), my_unsharp_mask(soft,1.0,3.0), my_unsharp_mask(soft,4.0,1.5)], ["soft input","a=0.5 s=1","a=1.5 s=1","a=3.0 s=1 (halos!)","a=1.5 s=4 (local contrast)"]
_cols = 5; _rows = -(-len(_imgs) // _cols)
fig, axes = plt.subplots(_rows, _cols, figsize=(3.0 * _cols, 3.0 * _rows), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

mask = g.astype(float) - cv2.GaussianBlur(g,(0,0),1.5)
_imgs, _titles = [g, np.clip(mask+128,0,255).astype(np.uint8), my_unsharp_mask(g,1.5,1.0)], ["input","the mask (f - blur), +128","input + 1.0 x mask"]
fig, axes = plt.subplots(1, 3, figsize=(4 * 3, 4), squeeze=False)
for ax, im, t in zip(axes.ravel(), _imgs, _titles):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255) if im.ndim == 2 else ax.imshow(im)
    ax.set_title(t, fontsize=10); ax.axis("off")
for ax in axes.ravel()[len(_imgs):]: ax.axis("off")
plt.tight_layout(); plt.show()

The `a=3.0` panel shows **halos**: overshoot at strong edges, the visual signature of
over-sharpening. Large sigma with modest amount does something different — local contrast
enhancement rather than edge sharpening. Two knobs, two distinct effects.

---
## Lightweight filtering for constrained devices

Your syllabus calls this out explicitly, and the ideas are all things you have now seen:

| technique | why it is cheap | cost |
|---|---|---|
| **separability** | k² multiplies → 2k | exact for Gaussian, none |
| **integral image / running sum** | box blur in O(1) per pixel, any radius | box only, no weighting |
| **repeated box** | 3 boxes ≈ Gaussian, still O(1) per pixel | ~1 level of error |
| **fixed-point / integer kernels** | no FPU needed (e.g. `[1,2,1]/4` as shifts) | rounding error |
| **power-of-two normalisation** | divide becomes a bit shift | forces kernel sums to 2ⁿ |
| **in-place / tiled processing** | fits in cache, no full-size temporaries | code complexity |
| **downscale, process, upscale** | 4× fewer pixels per halving | loses fine detail |

The classic integer Gaussian is `[[1,2,1],[2,4,2],[1,2,1]] / 16` — all integers, and the divide
by 16 is a 4-bit right shift. Verify it against the float version:

In [ ]:
intk = np.array([[1,2,1],[2,4,2],[1,2,1]], float)/16
fltk = my_getGaussianKernel(3, 0.85)
print("integer kernel /16:\n", (intk*16).astype(int))
d = np.abs(my_filter2D(g, intk) - my_filter2D(g, fltk))
print(f"\nmax difference vs the float Gaussian: {d.max():.2f} levels, mean {d.mean():.3f}")
print("kernel sum is a power of two:", int((intk*16).sum()), "-> divide is a >> 4 shift")

---
## Think about it

1. Your `my_filter2D` uses zero padding, so the border of a flat grey region comes out darker.
   Name two other padding choices and say which you would pick for (a) a microscope image on a
   black field, (b) a photograph. (Notebook 03 showed you the options.)
2. A box blur and a Gaussian of matched width both smooth. Blur a sharp edge with each and plot
   a horizontal slice through the result. Which one rings, and why does the Gaussian not?
3. Sharpening amplifies noise. Prove it: add mild Gaussian noise, then sharpen, and measure the
   noise standard deviation before and after. What is the standard fix? (Hint: the order of
   denoise and sharpen is not free.)
4. `my_filter2D(my_filter2D(img, k1), k2)` versus `my_filter2D(img, convolve2d_of_kernels)`.
   Convolution is associative, so these should agree. Verify it for two 3×3 kernels — and work
   out what size the combined kernel must be.
5. A 3×3 median filter costs 9 comparisons-ish per pixel and cannot be separated or done with an
   integral image. Why not? What is the standard trick for making a large-radius median fast?
   (Search: histogram-based median, constant-time median filtering.)

In [ ]:
# Scratch space.

---
## Stretch

1. **Integral image.** Write `integral(gray)` and then `box_blur_O1(gray, k)` that uses it, so the
   cost per pixel is 4 lookups regardless of `k`. Time it against `cv2.blur` at k=3, 15, 51.
2. **Bilateral filter.** Smooth *within* regions but not across edges, by weighting neighbours by
   both spatial and intensity distance. Compare against `cv2.bilateralFilter`. This is the basis
   of every "beauty mode".
3. **The full order-statistic family.** min, max, midpoint, alpha-trimmed mean. Show which noise
   type each one beats: pepper only, salt only, uniform, mixed.
4. **Laplacian sharpening vs unsharp.** `f - c*Laplacian(f)` is the other classical sharpener.
   Derive why it is (almost) the same operation as unsharp masking, then verify numerically.
5. **A fixed-point pipeline.** Redo `my_getGaussianKernel(5, 1.0)` as integers over a power-of-two
   denominator, do the whole convolution in `int32`, and report the maximum deviation from float.

In [ ]:
# Stretch scratch space.

---
## OpenCV API card - filtering

```python
dst = cv2.filter2D(src, ddepth, kernel, anchor=(-1,-1), delta=0, borderType=BORDER_DEFAULT)
```
Computes **correlation**, not convolution — flip the kernel yourself if you need true
convolution. `ddepth=-1` keeps the input depth and therefore **clips negatives**; pass
`cv2.CV_64F` (or `CV_32F`, `CV_16S`) to keep them.

```python
dst = cv2.blur(src, ksize)                              # box, ksize=(w,h)
dst = cv2.boxFilter(src, ddepth, ksize, normalize=True)
dst = cv2.GaussianBlur(src, ksize, sigmaX, sigmaY=0)    # ksize=(0,0) derives size from sigma
dst = cv2.medianBlur(src, ksize)                        # ksize ODD; >5 needs uint8
dst = cv2.bilateralFilter(src, d, sigmaColor, sigmaSpace)   # edge-preserving
dst = cv2.sepFilter2D(src, ddepth, kernelX, kernelY)    # separable, much faster
kernel1d = cv2.getGaussianKernel(ksize, sigma)          # (ksize, 1) COLUMN
```

```python
dst = cv2.Sobel(src, ddepth, dx, dy, ksize=3)     # dx=1,dy=0 for the x derivative
dst = cv2.Scharr(src, ddepth, dx, dy)             # better rotational symmetry at 3x3
dst = cv2.Laplacian(src, ddepth, ksize=1)
dst = cv2.erode(src, kernel) / cv2.dilate(src, kernel)      # notebook 12
```
Always use `ddepth=cv2.CV_64F` for derivative filters, then `cv2.convertScaleAbs` to display.